## Task 3 - Data Preprocessing and Feature Engineering
This section handles data loading, target discretisation (quantile-based), and feature engineering (ordinal and one-hot encoding).

In [8]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    classification_report,
)
from statsmodels.stats.contingency_tables import mcnemar

# Point Python at the scikit-eLCS GitHub master folder
ELCS_REPO = os.path.join(os.path.dirname(os.path.abspath('')), "scikit-eLCS-master")
if os.path.isdir(ELCS_REPO):
    sys.path.insert(0, ELCS_REPO)
from skeLCS import eLCS

In [9]:
RAW_DATA_PATH = "../ai_company_adoption.csv"
df = pd.read_csv(RAW_DATA_PATH)
print("Raw dataset shape:", df.shape)

# Quantile-based cutoffs for the target variable (Low, Medium, High)
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)

print("Class balances:")
print(df["productivity_class"].value_counts())

Raw dataset shape: (150000, 43)
Class balances:
productivity_class
1    50031
0    50023
2    49946
Name: count, dtype: int64


In [10]:
# Build the Preprocessed + Feature-Engineered Matrix
SELECTED_FEATURE_COLUMNS = [
    "ai_adoption_rate", "ai_maturity_score", "ai_failure_rate",
    "ai_training_hours", "task_automation_rate", "ai_adoption_stage", "industry",
]

X_df = df[SELECTED_FEATURE_COLUMNS].copy()

# Ordinal encoding for adoption stage
STAGE_ORDER = {"none": 0, "pilot": 1, "partial": 2, "full": 3}
X_df["ai_adoption_stage"] = X_df["ai_adoption_stage"].map(STAGE_ORDER)

# One-hot encoding for industry
X_df = pd.get_dummies(X_df, columns=["industry"], prefix="industry", dtype=float)

FEATURE_NAMES = list(X_df.columns)
X = X_df.values.astype(float)
y = df["productivity_class"].values.astype(int)

print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (150000, 15) | y shape: (150000,)


## Task 4 - Original and Improved eLCS Models (10-Seed Loop)
Running the 10-seed repeated evaluation for both the original eLCS baseline and the improved eLCS system.

In [11]:
# Repeated-Evaluation Settings
N_SEEDS = 10
TRAIN_SUBSAMPLE_SIZE = 1000
TEST_EVAL_SIZE = 5000
POPULATION_SIZE_N = round(1.5 * TRAIN_SUBSAMPLE_SIZE)
LEARNING_ITERATIONS = 5 * TRAIN_SUBSAMPLE_SIZE

print(f"Population size N = {POPULATION_SIZE_N}, learning_iterations = {LEARNING_ITERATIONS}")

Population size N = 1500, learning_iterations = 5000


In [ ]:
print("--- Training Original eLCS (Preprocessed) ---")
orig_per_seed_acc = []
orig_all_true = []
orig_all_pred = []
orig_last_model = None

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    rng = np.random.RandomState(seed)
    train_subsample_idx = rng.choice(len(X_train), TRAIN_SUBSAMPLE_SIZE, replace=False)
    
    X_train_sub = X_train[train_subsample_idx]
    y_train_sub = y_train[train_subsample_idx]
    X_test_eval = X_test[:TEST_EVAL_SIZE]
    y_test_eval = y_test[:TEST_EVAL_SIZE]

    model = eLCS(
        learning_iterations=LEARNING_ITERATIONS,
        N=POPULATION_SIZE_N,
        random_state=seed,
    )
    model.fit(X_train_sub, y_train_sub)

    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    
    orig_per_seed_acc.append(acc)
    orig_all_true.extend(y_test_eval.tolist())
    orig_all_pred.extend(y_pred.tolist())
    orig_last_model = model

    print(f"Seed {seed}: accuracy = {acc:.4f}")

--- Training Original eLCS (Preprocessed) ---


In [ ]:
print("--- Training Improved eLCS ---")
imp_per_seed_acc = []
imp_all_pred = []
imp_last_model = None

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    rng = np.random.RandomState(seed)
    train_subsample_idx = rng.choice(len(X_train), TRAIN_SUBSAMPLE_SIZE, replace=False)
    
    X_train_sub = X_train[train_subsample_idx]
    y_train_sub = y_train[train_subsample_idx]
    X_test_eval = X_test[:TEST_EVAL_SIZE]
    y_test_eval = y_test[:TEST_EVAL_SIZE]

    model = eLCS(
        learning_iterations=LEARNING_ITERATIONS,
        N=POPULATION_SIZE_N,
        do_correct_set_subsumption=True,
        random_state=seed,
    )
    model.fit(X_train_sub, y_train_sub)

    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    
    imp_per_seed_acc.append(acc)
    imp_all_pred.extend(y_pred.tolist())
    imp_last_model = model

    print(f"Seed {seed}: accuracy = {acc:.4f}")

## Task 5 - Decision Tree Baseline
Establishing a benchmark performance using a standard Decision Tree Classifier across the same 10-seed splits.

In [ ]:
print("--- Training Decision Tree Baseline ---")
dt_per_seed_acc = []
dt_all_pred = []

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    rng = np.random.RandomState(seed)
    train_subsample_idx = rng.choice(len(X_train), TRAIN_SUBSAMPLE_SIZE, replace=False)
    
    X_train_sub = X_train[train_subsample_idx]
    y_train_sub = y_train[train_subsample_idx]
    X_test_eval = X_test[:TEST_EVAL_SIZE]
    y_test_eval = y_test[:TEST_EVAL_SIZE]

    dt = DecisionTreeClassifier(random_state=seed)
    dt.fit(X_train_sub, y_train_sub)

    y_pred = dt.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    
    dt_per_seed_acc.append(acc)
    dt_all_pred.extend(y_pred.tolist())

    print(f"Seed {seed}: accuracy = {acc:.4f}")

## Task 6 - Evaluation, Statistical Testing, and Export
Aggregating classification reports for the improved model vs the baseline and executing McNemar's Test on paired predictions.

In [ ]:
orig_all_true = np.array(orig_all_true)
imp_all_pred = np.array(imp_all_pred)
dt_all_pred = np.array(dt_all_pred)

print("Classification report (Improved eLCS, pooled across all seeds):")
print(classification_report(
    orig_all_true, imp_all_pred, 
    target_names=["Low", "Medium", "High"], zero_division=0
))

print("\nClassification report (Decision Tree Baseline, pooled across all seeds):")
print(classification_report(
    orig_all_true, dt_all_pred, 
    target_names=["Low", "Medium", "High"], zero_division=0
))

# McNemar's Test: Improved eLCS vs Decision Tree
correct_imp = (imp_all_pred == orig_all_true)
correct_dt = (dt_all_pred == orig_all_true)

table = np.zeros((2, 2))
table[0, 0] = np.sum(correct_imp & correct_dt)
table[0, 1] = np.sum(correct_imp & ~correct_dt)
table[1, 0] = np.sum(~correct_imp & correct_dt)
table[1, 1] = np.sum(~correct_imp & ~correct_dt)

mcnemar_res = mcnemar(table, exact=False, correction=True)
print(f"\nMcNemar's test (Improved eLCS vs Decision Tree): statistic={mcnemar_res.statistic:.3f}, p-value={mcnemar_res.pvalue:.4g}")

In [ ]:
# Export eLCS rule population
imp_last_model.export_final_rule_population(
    headerNames=np.array(FEATURE_NAMES),
    className="productivity_class",
    filename="task6_elcs_improved_rules.csv",
    DCAL=True,
)
print("Exported final rule population to task6_elcs_improved_rules.csv")

# Export comparative seed results
seed_results = pd.DataFrame({
    "seed": list(range(N_SEEDS)),
    "model_original_elcs_acc": orig_per_seed_acc,
    "model_improved_elcs_acc": imp_per_seed_acc,
    "model_decision_tree_acc": dt_per_seed_acc,
})
seed_results.to_csv("task6_seed_results.csv", index=False)
print("Exported experimental seed comparisons to task6_seed_results.csv")